# Lab 4 — Building a Healthcare Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/angeloziletti/nlp-cas-dsmh/blob/main/notebooks/lab4_agent.ipynb)

In this lab you build an **agent**: a language model that uses **tools** and runs in a **loop** to answer questions — here, plain-language questions about a hospital patient database.

This is the **text-to-SQL** example from the Block G lecture, made real.

**How to use it**
- Run each cell in order (`Shift + Enter`).
- Read the printed **agent transcript** — it shows every step of the reason→act→observe loop.
- Cells marked **🔧 YOUR TURN** are for you to change a value and re-run.

**What you will build**
1. **Explore** the patient database the agent will query
2. The agent's **tools** — and a read-only safety guardrail
3. The **agent loop** — reason → act → observe → repeat
4. Watch it **work** through real questions, choosing its own steps
5. Watch it **fail** — so the fragility is real, not theoretical

**By design, this agent can only read the database — never change it.** No real patient data is used; the database is synthetic.

## Setup

Run the cells below. Your Gemini API key must be available:

- **In Colab:** stored as a Secret named `GEMINI_API_KEY` (the 🔑 icon in the left sidebar, with *Notebook access* on).
- **Running locally:** set as an environment variable before launching Jupyter (`export GEMINI_API_KEY=...` on macOS/Linux, `$env:GEMINI_API_KEY = "..."` on Windows PowerShell).

In [1]:
!pip install -q google-genai

In [2]:
import json
import os
import sqlite3
import urllib.request
import pandas as pd
from google import genai
from google.genai import types

# --- Read the Gemini API key (Colab Secrets, or local .env / env var) ---
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not found. In Colab, store it as a Secret named "
        "GEMINI_API_KEY. Locally, copy .env.example to .env and paste your "
        "key there (or set the environment variable before launching Jupyter)."
    )

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.1-flash-lite"

# --- Course data lives in the GitHub repo ---
BASE_URL = "https://raw.githubusercontent.com/angeloziletti/nlp-cas-dsmh/main/data/"

print("Setup complete. Model:", MODEL)

Setup complete. Model: gemini-3.1-flash-lite


---
## Part 1 — The patient database

Our agent answers questions about a hospital database. We are **given** the database — a small SQLite file — and, before building anything, our first job is the same one the agent will face: **understand what is in it.**

This is synthetic data. No real patient is involved.

In [3]:
# Download the database file and connect to it.
urllib.request.urlretrieve(BASE_URL + "hospital.db", "hospital.db")
conn = sqlite3.connect("hospital.db")

tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")]
print("Tables in the database:", tables)

Tables in the database: ['patients', 'admissions', 'diagnoses']


### Exploring an unfamiliar database

Nobody handed us a description of this database, so we explore it ourselves. First the **schema** — what columns each table has.

In [4]:
for t in tables:
    cols = conn.execute(f"PRAGMA table_info({t})").fetchall()
    print(f"{t}: " + ", ".join(f"{c[1]} ({c[2]})" for c in cols))

patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)


Next, a look at the actual **rows** — a few from each table.

In [5]:
for t in tables:
    print(f"\n{t}:")
    display(pd.read_sql(f"SELECT * FROM {t} LIMIT 4", conn))


patients:


,patient_id,age,sex
0,1,34,M
1,2,55,M
2,3,48,M
3,4,33,M



admissions:


,admission_id,patient_id,department,admission_date
0,1,37,Neurology,2024-04-21
1,2,32,Oncology,2024-11-15
2,3,10,Neurology,2024-03-08
3,4,36,General Medicine,2024-05-24



diagnoses:


,diagnosis_id,patient_id,diagnosis_name
0,1,1,Atrial Fibrillation
1,2,1,Heart Failure
2,3,2,COPD
3,4,3,Pneumonia


Finally — and this matters — the **values** inside the text columns. A column named `diagnosis_name` does not tell you *how* a diagnosis is actually written, so we check.

In [6]:
print("Departments:", [r[0] for r in conn.execute("SELECT DISTINCT department FROM admissions")])
print("Diagnoses:  ", [r[0] for r in conn.execute("SELECT DISTINCT diagnosis_name FROM diagnoses")])
print("Row counts: ", {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tables})

Departments: ['Neurology', 'Oncology', 'General Medicine', 'Pulmonology', 'Cardiology']
Diagnoses:   ['Atrial Fibrillation', 'Heart Failure', 'COPD', 'Pneumonia', 'Hypertension', 'Chronic Kidney Disease', 'Type 2 Diabetes', 'Asthma']
Row counts:  {'patients': 40, 'admissions': 60, 'diagnoses': 78}


You just did, by hand, what a careful data assistant does first: list the tables, read the schema, sample the rows, and check what the values actually look like. Note that diabetes is recorded as **`"Type 2 Diabetes"`**, not `"diabetes"` — hold on to that.

The **agent** you are about to build will do exactly this kind of exploration — but on its own, inside a loop. It is **not handed the schema**; its first tool, `get_schema`, is step one above, automated.

> **A note on real-world EHR data.** Real hospital databases are far larger than our three tables. Many are stored in a **standardized model** such as the **OMOP Common Data Model** — dozens of linked tables, with every clinical fact (diagnoses, drugs, procedures) coded to a controlled vocabulary rather than written as plain text. A query that looks simple here would, on OMOP, become a join through a vocabulary table. But the agent's job does not change: *the loop is identical — only the schema it has to discover is bigger.*

---
## Part 2 — The agent's tools

An agent is a model plus **tools** (Block G). Our agent gets two:

- **`get_schema()`** — returns the tables and columns. No input.
- **`run_sql(query)`** — runs one SQL `SELECT` query and returns the result *or an error message*.

Each is just a normal Python function.

In [7]:
def get_schema():
    cur = conn.cursor()
    tables = [r[0] for r in cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
    lines = []
    for t in tables:
        cols = cur.execute(f"PRAGMA table_info({t})").fetchall()
        lines.append(f"Table {t}: " + ", ".join(f"{c[1]} ({c[2]})" for c in cols))
    return "\n".join(lines)

print(get_schema())

Table patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
Table admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
Table diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)


In [8]:
def run_sql(query):
    # GUARDRAIL: this agent is read-only - reject anything that is not a SELECT.
    if not query.strip().lower().startswith("select"):
        return "ERROR: only SELECT queries are allowed (this agent is read-only)."
    try:
        cur = conn.execute(query)
        rows = cur.fetchall()
    except Exception as e:
        # The error is returned as text, NOT raised - so the agent can read and recover from it.
        return f"SQL ERROR: {e}"
    cols = [d[0] for d in cur.description]
    if not rows:
        return f"columns {cols}, 0 rows"
    return f"columns {cols}, {len(rows)} rows: {rows[:50]}"

# A working query:
print(run_sql("SELECT COUNT(*) FROM patients"))

columns ['COUNT(*)'], 1 rows: [(40,)]


In [9]:
# The guardrail and the error behaviour, side by side:
print("Non-SELECT query :", run_sql("DELETE FROM patients"))
print("Broken query     :", run_sql("SELECT * FROM no_such_table"))

Non-SELECT query : ERROR: only SELECT queries are allowed (this agent is read-only).
Broken query     : SQL ERROR: no such table: no_such_table


Two things to notice:

- A non-`SELECT` query is **refused** — the read-only guardrail (Block G). The agent *cannot* change the database even if it tried.
- A broken query does **not crash** — it returns a `SQL ERROR: ...` string. That is deliberate: the error becomes something the agent can **read and recover from**. Hold on to that — it is the next idea.

---
## Part 3 — The agent loop

Now the agent itself: a **loop** around the model (Block G). At each step the model returns a small JSON object —

```
{"thought": "...", "action": "...", "action_input": "..."}
```

— where `action` is `get_schema`, `run_sql`, or `final_answer`. We run the chosen tool, feed the result back, and loop. **Reason → act → observe → repeat.**

First, the agent's standing instructions — its **system prompt** (Block G: prompting as infrastructure).

In [10]:
SYSTEM_PROMPT = """You are a careful data assistant for a hospital patient database.
You answer questions by querying the database with SQL. You never guess.

You work step by step. At each step, respond with ONLY a JSON object:
  {"thought": "your reasoning", "action": "...", "action_input": "..."}

The action must be one of:
  - "get_schema"   : see the tables and columns. Use "" as action_input.
  - "run_sql"      : run one SQL SELECT query. action_input is the query.
  - "final_answer" : give the user the answer. action_input is plain-language text.

Rules:
  - Always call get_schema before writing any SQL.
  - Before filtering on a text column, consider checking which values it actually
    contains (e.g. SELECT DISTINCT ...), because values may not be worded as expected.
  - If run_sql returns an error, read it and try a corrected query.
  - Base your final answer only on query results - never on assumptions.
"""

In [11]:
TOOLS = {"get_schema": lambda _: get_schema(), "run_sql": run_sql}

def ask_model(question, transcript):
    prompt = (SYSTEM_PROMPT + f"\n\nQuestion: {question}\n\n" + transcript
              + "\nYour next step (JSON only):")
    response = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json"))
    return json.loads(response.text)

def run_agent(question, max_steps=6):
    print(f"QUESTION: {question}\n")
    transcript = ""
    for step in range(1, max_steps + 1):
        move = ask_model(question, transcript)
        thought = move["thought"]
        action = move["action"]
        action_input = move.get("action_input", "")

        print(f"--- step {step} ---")
        print(f"  THOUGHT: {thought}")
        print(f"  ACTION:  {action}  {action_input}")

        if action == "final_answer":
            print(f"\nFINAL ANSWER: {action_input}")
            return action_input

        observation = TOOLS[action](action_input)
        print(f"  OBSERVATION: {observation}\n")
        transcript += (f"Thought: {thought}\nAction: {action}\n"
                       f"Action input: {action_input}\nObservation: {observation}\n\n")

    print("\n[stopped: reached max_steps without a final answer]")
    return None

One subtle but important point (Block G): **the model never touches the database itself.** It only *returns a request* — a JSON object naming an action. The loop above is what actually runs the tool. That separation — the model **decides**, the system **executes** — is exactly what makes the read-only guardrail possible: the model cannot run anything that `run_sql` refuses to.

---
## Part 4 — Watch it work

Run the agent and **read the transcript**. Every `step` block is one turn of the loop: a thought, an action (a tool call), and the observation that came back.

In [12]:
run_agent("How many patients are in the database?")

QUESTION: How many patients are in the database?

--- step 1 ---
  THOUGHT: I need to determine the number of patients in the database. First, I will inspect the schema to identify the correct table.
  ACTION:  get_schema  
  OBSERVATION: Table patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
Table admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
Table diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)

--- step 2 ---
  THOUGHT: The table 'patients' likely contains all patient records. I will count the number of rows in the 'patients' table to find the total number of patients.
  ACTION:  run_sql  SELECT COUNT(*) FROM patients
  OBSERVATION: columns ['COUNT(*)'], 1 rows: [(40,)]

--- step 3 ---
  THOUGHT: The query result indicates there are 40 patients in the database.
  ACTION:  final_answer  There are 40 patients in the database.

FINAL ANSWER: There are 40 patients in the database.


'There are 40 patients in the database.'

Even this simple question took **more than one step**: the agent first called `get_schema` to learn the structure, *then* wrote the SQL. It was not told to do that — it decided. That is the loop.

In [13]:
run_agent("What is the average age of patients admitted to the Cardiology department?")

QUESTION: What is the average age of patients admitted to the Cardiology department?

--- step 1 ---
  THOUGHT: I need to find the average age of patients in the Cardiology department. First, I need to see the schema to identify the relevant tables for patients, admissions, and departments.
  ACTION:  get_schema  
  OBSERVATION: Table patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
Table admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
Table diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)

--- step 2 ---
  THOUGHT: I need to check the distinct values in the 'department' column to ensure I filter correctly for 'Cardiology'.
  ACTION:  run_sql  SELECT DISTINCT department FROM admissions
  OBSERVATION: columns ['department'], 5 rows: [('Neurology',), ('Oncology',), ('General Medicine',), ('Pulmonology',), ('Cardiology',)]

--- step 3 ---
  THOUGHT: Now that I have verified 'Cardiology' exists in the

'The average age of patients admitted to the Cardiology department is 57.125 years.'

This one needs a **join** across two tables. Watch how the agent built up to it across steps — and if it made a wrong query along the way, see how the error came back as an observation and it tried again. **That self-correction is what an agent can do that a single prompt cannot.**

In [14]:
run_agent("Which department had the most admissions?")

QUESTION: Which department had the most admissions?

--- step 1 ---
  THOUGHT: I need to find which department has the most admissions. First, I need to understand the database schema to locate the relevant tables.
  ACTION:  get_schema  
  OBSERVATION: Table patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
Table admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
Table diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)

--- step 2 ---
  THOUGHT: I need to count the number of admissions per department and identify the department with the highest count.
  ACTION:  run_sql  SELECT department, COUNT(*) as admission_count FROM admissions GROUP BY department ORDER BY admission_count DESC LIMIT 1
  OBSERVATION: columns ['department', 'admission_count'], 1 rows: [('Cardiology', 16)]

--- step 3 ---
  THOUGHT: I have identified the department with the highest number of admissions using the count aggregation.
  

'The Cardiology department had the most admissions, with 16 recorded.'

In [15]:
# 🔧 YOUR TURN: ask the agent your own question about the database.
run_agent("How many patients are older than 70?")

QUESTION: How many patients are older than 70?

--- step 1 ---
  THOUGHT: I need to determine the schema of the database to identify the table and column names related to patients and their ages.
  ACTION:  get_schema  
  OBSERVATION: Table patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
Table admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
Table diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)

--- step 2 ---
  THOUGHT: I will count the number of patients where the 'age' column is greater than 70 from the 'patients' table.
  ACTION:  run_sql  SELECT COUNT(*) FROM patients WHERE age > 70
  OBSERVATION: columns ['COUNT(*)'], 1 rows: [(6,)]

--- step 3 ---
  THOUGHT: The query result shows there are 6 patients older than 70.
  ACTION:  final_answer  There are 6 patients older than 70.

FINAL ANSWER: There are 6 patients older than 70.


'There are 6 patients older than 70.'

---
## Part 5 — Error recovery

Look back at the transcripts above. Whenever a query failed, `run_sql` returned a `SQL ERROR: ...` string — and that string became the agent's next **observation**, something it could read and act on.

Here is what that looks like inside a loop — the agent gets a column name wrong, sees the error, and corrects itself:

```
step 2  ACTION: run_sql   SELECT AVG(age) FROM patients WHERE department = 'Cardiology'
        OBSERVATION: SQL ERROR: no such column: department
step 3  ACTION: run_sql   SELECT AVG(p.age) FROM patients p
                          JOIN admissions a ON p.patient_id = a.patient_id
                          WHERE a.department = 'Cardiology'
        OBSERVATION: columns ['AVG(p.age)'], 1 rows: [(58.3,)]
```

This is why, in Part 2, the broken query returned *text* instead of crashing. **An error is not the end of the loop — it is just another observation.** A single, one-shot prompt has no equivalent: it produces one query and stops. Noticing a mistake and trying again is one of the few things an agent genuinely adds.

---
## Part 6 — Break it on purpose

An agent is powerful — and, as Block G stressed, the hardest thing to keep reliable. Let's see two real failure modes.

**Failure 1 — the confident wrong answer.** In our data, diabetes is recorded as `"Type 2 Diabetes"`. Watch what an exact-match query does:

In [16]:
print("Querying for 'diabetes' (exact match):")
print("  ", run_sql("SELECT COUNT(*) FROM diagnoses WHERE diagnosis_name = 'diabetes'"))

print("\nQuerying for '%Diabetes%' (the value as actually stored):")
print("  ", run_sql("SELECT COUNT(*) FROM diagnoses WHERE diagnosis_name LIKE '%Diabetes%'"))

Querying for 'diabetes' (exact match):
   columns ['COUNT(*)'], 1 rows: [(0,)]

Querying for '%Diabetes%' (the value as actually stored):
   columns ['COUNT(*)'], 1 rows: [(7,)]


The first query returns **0** — and **no error**. If an agent writes that query and skips checking the actual values, it will report *"no patients have diabetes"* — fluently, confidently, and **wrong**.

This is the danger from Block D and Block H: a wrong answer that looks exactly like a right one. An agent does not remove that risk — it can deliver it faster, wrapped in a tidy transcript. (Our system prompt *tells* the agent to check distinct values first — but nothing guarantees it always will.)

**🔧 YOUR TURN — the system prompt is a safety mechanism.** The agent's `SYSTEM_PROMPT` (Part 3) contains the rule *"Before filtering on a text column, consider checking which values it actually contains."* That single line is the agent's guardrail against the trap above. Scroll up to the `SYSTEM_PROMPT` cell, **delete that rule** (or the *"Always call get_schema..."* rule), re-run that cell, then re-run an agent question. Watch how the behaviour shifts. Instructions shape what the agent does — though, as you may see, not always perfectly predictably.

**Failure 2 — a question the data cannot answer.** Our database has no medication information.

In [17]:
run_agent("What medications is patient 5 taking?")

QUESTION: What medications is patient 5 taking?

--- step 1 ---
  THOUGHT: I need to find the medications associated with patient 5. First, I will retrieve the database schema to understand the table structure.
  ACTION:  get_schema  
  OBSERVATION: Table patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
Table admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
Table diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)

--- step 2 ---
  THOUGHT: I need to find the table that contains medication information. Looking at the schema, none of the tables (patients, admissions, diagnoses) explicitly mention medications. I will list all tables again to ensure I didn't miss anything, or perhaps I need to check if there is a 'medications' table that was not returned. Wait, the provided schema shows only 3 tables. I should verify if I have full schema access.
  ACTION:  get_schema  
  OBSERVATION: Table patients: pat

'I am sorry, but the hospital database provided does not contain any information regarding medications.'

Inspect what the agent did. A well-behaved agent calls `get_schema`, sees there is no medication table, and **says so**. A poorly behaved one would invent an answer, or loop trying. Either way: the agent is only as good as its data and its guardrails.

**The `max_steps` guardrail.** The loop must never run forever (Block G). Here we cap it deliberately low and watch it stop before finishing:

In [18]:
run_agent("What is the average age of patients in each department?", max_steps=2)

QUESTION: What is the average age of patients in each department?

--- step 1 ---
  THOUGHT: I need to find the average age of patients grouped by their department. First, I need to see the schema to identify the relevant tables and columns.
  ACTION:  get_schema  
  OBSERVATION: Table patients: patient_id (INTEGER), age (INTEGER), sex (TEXT)
Table admissions: admission_id (INTEGER), patient_id (INTEGER), department (TEXT), admission_date (TEXT)
Table diagnoses: diagnosis_id (INTEGER), patient_id (INTEGER), diagnosis_name (TEXT)

--- step 2 ---
  THOUGHT: I need to join the 'patients' table with the 'admissions' table on 'patient_id' and then group by 'department' to calculate the average age.
  ACTION:  run_sql  SELECT T2.department, AVG(T1.age) FROM patients AS T1 JOIN admissions AS T2 ON T1.patient_id = T2.patient_id GROUP BY T2.department
  OBSERVATION: columns ['department', 'AVG(T1.age)'], 5 rows: [('Cardiology', 57.125), ('General Medicine', 49.42857142857143), ('Neurology', 54.

The agent ran out of steps and stopped cleanly — instead of looping indefinitely. A small guardrail, but an essential one.

---
## Part 7 — Was an agent the right tool?

You just built an agent. Block G's closing question still applies: **was it worth it?**

- For *"How many patients are in the database?"* — an agent is **overkill**. That is one fixed query; a single line of SQL is simpler, faster, and predictable.
- For *open-ended questions over a database you don't know in advance* — where the path depends on what the schema and the data turn out to be — the loop genuinely earns its place.

> The rule from the lecture: use the **simplest thing that works**. An agent is the right tool only when the task truly needs **choice, tools, and iteration**.

---
## Recap — what you built

| Part | What you did |
|---|---|
| Database | A small synthetic patient database |
| Tools | `get_schema` and `run_sql` — with a read-only guardrail |
| The loop | An agent that reasons, acts, observes, and repeats |
| Watch it work | Saw it choose its own steps and recover from errors |
| Break it | Saw a confident wrong answer, a question it couldn't answer, and the step cap |

**You have built the agent from the Block G lecture — a model that uses tools and takes steps.**

That completes the four labs of this module. In the final block, we turn to the question that decides whether any of this is usable: **how do you know it actually works?**